# Kinematics Metrics — Normalized Jerk

Demonstrates `kinema.kinematics.metrics` on real bouldering COM data.

**Contents**
1. Load COM from Parquet (written by notebook 01)
2. Normalized jerk — formula, full-clip score
3. Rolling-window jerk — smoothness variation over time
4. Segment comparison — first half vs second half of clip
5. Synthetic sanity check — constant velocity vs jittery trajectory
6. `segment_jerk` vs `normalized_jerk` equivalence

**Formula (Balasubramanian et al. 2015)**

$$\text{NJ} = \sqrt{\frac{1}{2} \int \|\mathbf{j}(t)\|^2 \, dt \cdot \frac{T^5}{L^2}}$$

where $\mathbf{j}(t)$ is the 3-D jerk vector (third derivative of COM position),
$T$ is duration, $L$ is path length.  **Lower NJ = smoother movement.**

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from kinema.io.com import read_com
from kinema.kinematics.metrics import normalized_jerk, segment_jerk, MIN_FRAMES

DATA_DIR = Path('../data/processed')

# Must match notebook 01 output
COM_PATH = DATA_DIR / 'VID-20260528-WA0044_com.parquet'
FPS = 30.0

print(f'MIN_FRAMES threshold : {MIN_FRAMES}')
print(f'COM file             : {COM_PATH}')

## 1. Load COM

In [ ]:
com = read_com(COM_PATH)
com_valid = com.dropna(subset=['com_x', 'com_y', 'com_z']).reset_index(drop=True)

t = com_valid['timestamp_ms'].to_numpy() / 1000.0  # seconds
duration = t[-1] - t[0]

print(f'Frames total    : {len(com)}')
print(f'Frames with COM : {len(com_valid)}')
print(f'Duration        : {duration:.2f} s')
print()
com_valid.head()

## 2. Full-clip normalized jerk

In [ ]:
nj_full = normalized_jerk(com_valid, FPS)

# Path length for reference
dx = np.diff(com_valid['com_x'].to_numpy(dtype=float))
dy = np.diff(com_valid['com_y'].to_numpy(dtype=float))
dz = np.diff(com_valid['com_z'].to_numpy(dtype=float))
path_length = float(np.sum(np.sqrt(dx**2 + dy**2 + dz**2)))

print(f'Normalized jerk (full clip) : {nj_full:.4f}')
print(f'Duration                    : {duration:.2f} s')
print(f'Path length (normalized)    : {path_length:.4f}')
print()
print('Interpretation: lower NJ = smoother COM movement.')
print('Typical bouldering clips range 0.1–2.0 depending on move style.')

## 3. Rolling-window jerk

Compute NJ in overlapping windows (~1 s each) to see how smoothness evolves over the clip.
Windows with fewer than `MIN_FRAMES` frames are skipped.

In [ ]:
WINDOW_FRAMES = int(FPS * 1.0)   # 1-second window
STEP_FRAMES   = int(FPS * 0.25)  # 250 ms step (75% overlap)

n = len(com_valid)
window_nj   = []
window_tmid = []

for start in range(0, n - WINDOW_FRAMES + 1, STEP_FRAMES):
    end = start + WINDOW_FRAMES - 1
    nj_w = segment_jerk(com_valid, start, end, FPS)
    if not np.isnan(nj_w):
        t_mid = (t[start] + t[end]) / 2
        window_nj.append(nj_w)
        window_tmid.append(t_mid)

window_nj   = np.array(window_nj)
window_tmid = np.array(window_tmid)

print(f'Windows computed : {len(window_nj)}')
print(f'NJ range         : {window_nj.min():.4f} – {window_nj.max():.4f}')
print(f'NJ median        : {np.median(window_nj):.4f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# COM trajectory (y, flipped so up = higher on wall)
ax = axes[0]
ax.plot(t, 1 - com_valid['com_y'], color='steelblue', linewidth=1.2, label='COM y (smoothed)')
ax.set_ylabel('Height (1 - com_y)')
ax.set_title('COM vertical position')
ax.legend(fontsize=9)

# COM x
ax = axes[1]
ax.plot(t, com_valid['com_x'], color='steelblue', linewidth=1.2, label='COM x (smoothed)')
ax.set_ylabel('Horizontal (com_x)')
ax.set_title('COM horizontal position')
ax.legend(fontsize=9)

# Rolling NJ
ax = axes[2]
ax.axhline(nj_full, color='gray', linestyle='--', linewidth=0.9, label=f'full-clip NJ = {nj_full:.3f}')
sc = ax.scatter(window_tmid, window_nj, c=window_nj, cmap='RdYlGn_r',
                s=18, zorder=5, vmin=window_nj.min(), vmax=np.percentile(window_nj, 95))
ax.plot(window_tmid, window_nj, color='gray', linewidth=0.5, alpha=0.4)
plt.colorbar(sc, ax=ax, label='NJ (higher = jerkier)', pad=0.01)
ax.set_ylabel('Normalized Jerk')
ax.set_xlabel('Time (s)')
ax.set_title(f'Rolling NJ  (window = {WINDOW_FRAMES/FPS:.2f} s, step = {STEP_FRAMES/FPS:.2f} s)')
ax.legend(fontsize=9)

plt.suptitle('COM Trajectory & Rolling Normalized Jerk', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Segment comparison — first half vs second half

In [ ]:
mid = len(com_valid) // 2

nj_first  = segment_jerk(com_valid, 0,   mid - 1,          FPS)
nj_second = segment_jerk(com_valid, mid, len(com_valid) - 1, FPS)

print(f'Full clip  NJ : {nj_full:.4f}  ({len(com_valid)} frames, {duration:.2f} s)')
print(f'First half NJ : {nj_first:.4f}  ({mid} frames, {t[mid-1]-t[0]:.2f} s)')
print(f'Second half NJ: {nj_second:.4f}  ({len(com_valid)-mid} frames, {t[-1]-t[mid]:.2f} s)')
print()
smoother = 'first half' if nj_first < nj_second else 'second half'
print(f'→ {smoother} is smoother')

fig, ax = plt.subplots(figsize=(6, 4))
labels = ['Full clip', 'First half', 'Second half']
values = [nj_full, nj_first, nj_second]
colors = ['steelblue', 'mediumseagreen', 'tomato']
bars = ax.bar(labels, values, color=colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Normalized Jerk  (lower = smoother)')
ax.set_title('NJ: full clip vs halves')
plt.tight_layout()
plt.show()

## 5. Synthetic sanity check

Validate the metric on known inputs:
- **Constant velocity** — jerk = 0 → NJ ≈ 0
- **Smooth sine** — moderate jerk → NJ > 0
- **Jittery** — same path + random noise → NJ >> smooth

In [ ]:
def make_com(x: np.ndarray, y: np.ndarray, z: np.ndarray, fps: float = 30.0) -> pd.DataFrame:
    """Build a minimal COM DataFrame from coordinate arrays."""
    n = len(x)
    dt_ms = int(1000 / fps)
    return pd.DataFrame({
        'frame_idx':   np.arange(n, dtype=np.int32),
        'timestamp_ms': np.arange(n, dtype=np.int64) * dt_ms,
        'com_x': x.astype(np.float32),
        'com_y': y.astype(np.float32),
        'com_z': z.astype(np.float32),
    })


fps_synth = 30.0
n_synth   = 90   # 3 seconds
t_synth   = np.linspace(0, 3, n_synth)
rng       = np.random.default_rng(42)

# --- trajectories ---
com_const   = make_com(t_synth, np.zeros(n_synth),                         np.zeros(n_synth), fps_synth)
com_sine    = make_com(np.sin(2 * np.pi * t_synth), t_synth / 3,           np.zeros(n_synth), fps_synth)
com_jittery = make_com(
    np.sin(2 * np.pi * t_synth) + rng.normal(0, 0.15, n_synth),
    t_synth / 3                  + rng.normal(0, 0.05, n_synth),
    rng.normal(0, 0.05, n_synth),
    fps_synth,
)

nj_const   = normalized_jerk(com_const,   fps_synth)
nj_sine    = normalized_jerk(com_sine,    fps_synth)
nj_jittery = normalized_jerk(com_jittery, fps_synth)

print('Synthetic trajectory NJ values')
print(f'  Constant velocity : {nj_const:.6f}  (expect ≈ 0)')
print(f'  Smooth sine       : {nj_sine:.4f}')
print(f'  Jittery sine      : {nj_jittery:.4f}  (expect > smooth sine)')
print()
assert nj_const   < 0.01,          f'Expected near-zero NJ for constant velocity, got {nj_const}'
assert nj_jittery > nj_sine,       f'Expected jittery > smooth, got {nj_jittery} vs {nj_sine}'
print('Assertions passed.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

traj_data = [
    ('Constant velocity', com_const,   nj_const,   'steelblue'),
    ('Smooth sine',       com_sine,    nj_sine,    'mediumseagreen'),
    ('Jittery sine',      com_jittery, nj_jittery, 'tomato'),
]

for ax, (title, df, nj, color) in zip(axes, traj_data):
    ax.plot(df['com_x'], df['com_y'], color=color, linewidth=1.5, alpha=0.85)
    ax.scatter([df['com_x'].iloc[0]], [df['com_y'].iloc[0]],
               color='black', s=40, zorder=5, label='start')
    ax.scatter([df['com_x'].iloc[-1]], [df['com_y'].iloc[-1]],
               color='red', s=40, marker='*', zorder=5, label='end')
    ax.set_title(f'{title}\nNJ = {nj:.4f}', fontsize=10)
    ax.set_xlabel('com_x')
    ax.set_ylabel('com_y')
    ax.legend(fontsize=8)

plt.suptitle('Synthetic trajectories — NJ comparison', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. `segment_jerk` vs `normalized_jerk` equivalence

`segment_jerk(com, 0, len(com)-1, fps)` must equal `normalized_jerk(com, fps)` exactly.

In [ ]:
n = len(com_valid)
nj_via_segment = segment_jerk(com_valid, 0, n - 1, FPS)
nj_via_full    = normalized_jerk(com_valid, FPS)

print(f'normalized_jerk         : {nj_via_full:.10f}')
print(f'segment_jerk (0, n-1)   : {nj_via_segment:.10f}')
print(f'Difference              : {abs(nj_via_full - nj_via_segment):.2e}')
assert np.isclose(nj_via_full, nj_via_segment, rtol=1e-10), 'Mismatch!'
print()
print('Equivalence confirmed (rtol=1e-10).')

## 7. Short-segment edge case

Segments shorter than `MIN_FRAMES` must return `NaN`.

In [ ]:
for n_frames in range(1, MIN_FRAMES + 2):
    t_short = np.linspace(0, 1, n_frames)
    df_short = make_com(t_short, np.zeros(n_frames), np.zeros(n_frames))
    nj_val = normalized_jerk(df_short, 30.0)
    tag = 'NaN (expected)' if n_frames < MIN_FRAMES else 'valid'
    print(f'  n={n_frames:2d} → NJ = {nj_val}  [{tag}]')

print(f'\nMIN_FRAMES = {MIN_FRAMES}')